# CatBoost v5 — Per-Security, Horizon-Sweep

**Third benchmark notebook** in the Norwegian swap rate forecastability study.
Methodologically identical to `model_htboost_v5_clean.ipynb` and
`scripts/model_xgboost_v5_clean.py` in every respect except the model.
Logic reference: XGBoost script (most recently validated pipeline).
Format/structure reference: HTBoost notebook.

**Three documented, known divergences** (all stated in the thesis):

1. **Loss function** — CatBoost uses RMSE (squared error). HTBoost uses Student-t loss
   (`LOSS='t'`). The comparison is NOT a perfect isolation of the smooth-split mechanism.
   XGBoost also uses squared error; see XGBoost script docstring for the identical note.

2. **Bootstrap / subsampling** — CatBoost uses Minimum Variance Sampling (MVS, the CPU
   default) rather than Bernoulli random subsampling. MVS preferentially resamples
   observations with high gradient variance. The `subsample` grid parameter is therefore
   the MVS *data fraction*, not XGBoost-style random subsampling; see column
   `cat_mvs_fraction` in the tuning log. This is a second known divergence, analogous to
   the loss-function difference — we let CatBoost use its native mechanism.

3. **Tree structure** — CatBoost builds symmetric (oblivious) trees: all nodes at a given
   depth use the same feature and threshold. This is architecturally distinct from XGBoost's
   greedy asymmetric trees. `depth` values are NOT directly comparable across models (same
   depth integer ≠ same tree shape or flexibility); this is part of CatBoost's distinct
   character as a third comparison point.

Output schema is byte-identical to both HTBoost and XGBoost on `SHARED_COLS + pca_k +
xm_pca_evr`; model-specific diagnostic columns are `cat_n_trees`, `cat_depth`, `cat_lr`.

## Setup

Import the scientific-Python stack, statistical-test libraries, and CatBoost (no Julia
dependency). All shared modules are imported unchanged from `src/`.

In [ ]:
import re
import os
import sys
import json
import hashlib
import itertools
import pickle
import time
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy.stats import binomtest, norm
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
from sklearn.metrics import r2_score

# catboost 1.2.10 — pip-installed; not yet in environment.yml (tracked separately).
# Version pinned here for appendix reproducibility.
from catboost import CatBoostRegressor  # catboost==1.2.10

# ── Repo root resolution (same pattern as the HTBoost notebook) ───────────────
_ROOT = os.getcwd()
while _ROOT != os.path.dirname(_ROOT) and not os.path.isdir(os.path.join(_ROOT, 'src')):
    _ROOT = os.path.dirname(_ROOT)
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

# ── Shared modules — imported unchanged; no modifications to any src/ file ────
from src.data.bloomberg import load_data
from src.data.norway import load_norway_raw, print_connectivity_report
from src.features.macro import add_macro_features
from src.features.norway import add_norway_features
from src.features.panel import build_panel
from src.features.pca import fit_transform_xm_pca
from src.evaluation.metrics import (
    compute_metrics_row,
    SHARED_COLS,
    MTC_COLS,
    apply_mtc_family,
    finalize_long_csv,
)
import src.config as config
from src.config import MACHINE_ID

warnings.filterwarnings('ignore')
print(f'Imports OK  (catboost 1.2.10, pandas {pd.__version__}, numpy {np.__version__})')
print(f'MACHINE_ID = {MACHINE_ID!r}')
print(f'_ROOT      = {_ROOT!r}')

### Configuration

All run parameters in one place. Nothing here is chosen by looking at an OOS fold.
`SMOKE_MODE = True` (default): runs only NOR_10Y × H=21, writes to `_smoke/` subdirectory.
Set `SMOKE_MODE = False` for the full sweep.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CONFIG — all run parameters live here; nothing is selected on OOS performance
# ════════════════════════════════════════════════════════════════════════════

NOTEBOOK   = 'catboost_v5'      # short provenance label (stamped in every metrics row)
MODEL_KIND = 'per_security'     # same as HTBoost / XGBoost per-security notebooks
IS_POOLED  = False
RUN_TS     = datetime.now(tz=timezone.utc).isoformat()

# SMOKE_MODE: True → NOR_10Y × H=21 only (→ _smoke/); False → full sweep
SMOKE_MODE = True

# Horizon grid — identical to HTBoost / XGBoost
H_GRID      = config.H_GRID       # [1, 5, 21, 63]
H_GRID_LONG = [126, 252]          # 6m, 12m — gated on data length (same rule as HTBoost)

# Walk-forward folds — identical to HTBoost / XGBoost (from shared config)
WF_FOLDS   = config.WF_FOLDS
FOLD_NAMES = config.FOLD_NAMES

# Universe thresholds — identical to HTBoost / XGBoost
UNIVERSE_MIN_OBS = 500
MIN_TRAIN_OBS    = config.MIN_TRAIN_OBS   # 252
MIN_TEST_OBS     = config.MIN_TEST_OBS    # 20
TARGET_LAGS      = 5                       # AR lags of chg_1d — identical to HTBoost / XGBoost

# PCA compression of the cross-market xm_* block — identical to HTBoost / XGBoost
XM_PCA_ENABLE = config.XM_PCA_ENABLE
XM_PCA_VAR    = config.XM_PCA_VAR    # 0.95
XM_PCA_KMAX   = config.XM_PCA_KMAX   # 12

# Compressed fingerprint of the feature specification; used in config_hash
FEAT_SPEC = f'pca_var{XM_PCA_VAR}_kmax{XM_PCA_KMAX}_tl{TARGET_LAGS}'

# Output directory — SEPARATE from HTBoost (v5_nor/) and XGBoost (xgb_v5_nor/)
OUT_DIR   = os.path.join(_ROOT, 'results', 'catboost_v5_nor')
SMOKE_DIR = os.path.join(OUT_DIR, '_smoke')

# Norway feature cache (cache-first, same as XGBoost script — live=False)
NOR_CACHE = os.path.join(_ROOT, 'data', 'cache', 'norway_raw_features.csv')

# Smoke test defaults
SMOKE_SEC = 'NOR_10Y'
SMOKE_H   = 21

META_COLS = {'date', 'security', 'y', 'level'}

# ════════════════════════════════════════════════════════════════════════════
# CatBoost hyperparameter tuning
# Pre-committed grid; no OOS fold ever influences these values.
#
# KNOWN DIVERGENCE 1 (loss): CatBoost uses RMSE. HTBoost uses Student-t.
# KNOWN DIVERGENCE 2 (bootstrap): CatBoost uses MVS (Minimum Variance Sampling,
#   the CPU default). XGBoost uses Bernoulli random subsampling. The `subsample`
#   parameter here is the MVS *data fraction* — it is NOT equivalent to XGBoost's
#   `subsample` (Bernoulli). Labelled `cat_mvs_fraction` in the tuning log and
#   any reported hyperparameter table to prevent apples-to-oranges comparison.
# KNOWN DIVERGENCE 3 (tree structure): CatBoost builds symmetric (oblivious) trees.
#   `depth` [4, 6] is NOT directly comparable to XGBoost `max_depth` [3, 5]:
#   at equal depth integers, CatBoost's symmetric constraint produces different
#   tree shapes and different numbers of effective decision rules. This is part of
#   CatBoost's distinct character; we do not suppress it to force parity.
# ════════════════════════════════════════════════════════════════════════════

LOSS_FUNCTION  = 'RMSE'
N_TREES_MAX    = 500     # maximum boosting rounds (early stopping applied in inner CV)
EARLY_STOP_RND = 20      # patience: rounds without improvement before stopping
INNER_VAL_FRAC = 0.20    # time-ordered inner validation: last 20% of training window

# Pre-committed hyperparameter grid (tuned within training window only).
# 6 parameters × same factor counts as XGBoost → 3×2×2×2×3×2 = 144 combinations.
CAT_PARAM_GRID = {
    'learning_rate':      [0.01, 0.05, 0.10],
    # CatBoost oblivious-tree depth: all splits at a level share one feature/threshold.
    # Range [4, 6] is appropriate for this architecture; not equivalent to XGBoost depth.
    'depth':              [4, 6],
    # MVS data fraction (NOT Bernoulli subsampling — see KNOWN DIVERGENCE 2 above).
    # Logged as cat_mvs_fraction in the tuning CSV to prevent misreading.
    'subsample':          [0.8, 1.0],
    # rsm: random subspace method — fraction of features considered per split.
    # Direct analog of XGBoost colsample_bytree.
    'rsm':                [0.8, 1.0],
    # L2 leaf regularization — direct analog of XGBoost reg_lambda.
    'l2_leaf_reg':        [1.0, 5.0, 10.0],
    # Minimum number of training samples required in a leaf.
    # Closest analog of XGBoost min_child_weight (integer count vs hessian sum).
    'min_data_in_leaf':   [1, 5],
}
# Grid size: 3 × 2 × 2 × 2 × 3 × 2 = 144 combinations

print(f'SMOKE_MODE={SMOKE_MODE}  NOTEBOOK={NOTEBOOK!r}  MODEL_KIND={MODEL_KIND!r}')
print(f'H_GRID={H_GRID}  (+long {H_GRID_LONG} if data supports)')
print(f'LOSS_FUNCTION={LOSS_FUNCTION!r}  N_TREES_MAX={N_TREES_MAX}  '
      f'EARLY_STOP_RND={EARLY_STOP_RND}  INNER_VAL_FRAC={INNER_VAL_FRAC}')
print(f'PCA: enable={XM_PCA_ENABLE}  var≥{XM_PCA_VAR}  kmax={XM_PCA_KMAX}')
print(f'OUT_DIR={OUT_DIR!r}')
_n_combos = len(list(itertools.product(*CAT_PARAM_GRID.values())))
print(f'CatBoost param grid ({_n_combos} combinations):')
for _k, _v in CAT_PARAM_GRID.items():
    _note = '← MVS data fraction, NOT Bernoulli subsample' if _k == 'subsample' else ''
    print(f'  {_k:<20s}: {_v}  {_note}')

### Helper utilities

`_config_hash`, `_score`, `_prepare_x_cat` (column-name sanitization + ±inf→NaN),
`_done_secs` / `_append_csv` (checkpoint helpers identical to XGBoost script),
and file-path helpers (analogous to `V5_WF_CSV` etc. in the HTBoost notebook).

In [ ]:
def _config_hash(cfg, extra=''):
    """Stable 12-char MD5 fingerprint of a config dict. Identical logic to HTBoost notebook."""
    payload = (json.dumps({k: cfg.get(k) for k in sorted(cfg)},
                           sort_keys=True, default=str)
               + '|' + str(extra))
    return hashlib.md5(payload.encode()).hexdigest()[:12]


def _score(y, yhat):
    """DirAcc / R² / n_obs helper for diagnostic printing."""
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    m = np.isfinite(y) & np.isfinite(yhat)
    if m.sum() < 5:
        return None
    return {'dir_acc': float(np.mean(np.sign(y[m]) == np.sign(yhat[m]))),
            'r2':      float(r2_score(y[m], yhat[m])),
            'n_obs':   int(m.sum())}


def _prepare_x_cat(df):
    """Sanitize column names and replace ±inf with NaN.
    CatBoost handles NaN natively for tree-building (no fillna needed).
    The xmpca_* block is already zero-filled by fit_transform_xm_pca."""
    df = df.copy()
    rmap = {c: re.sub(r'[^a-zA-Z0-9_]', '_', str(c)).strip('_') for c in df.columns}
    df = df.rename(columns=rmap)
    # Deduplicate any name collisions created by sanitization (rare with these features)
    seen, new_cols = {}, []
    for c in df.columns:
        if c in seen:
            seen[c] += 1
            new_cols.append(f'{c}_{seen[c]}')
        else:
            seen[c] = 0
            new_cols.append(c)
    df.columns = new_cols
    return df.replace([np.inf, -np.inf], np.nan).astype(np.float64)


def _done_secs(csv_path):
    """Securities whose full fold-batch is already written to csv_path."""
    if not os.path.exists(csv_path):
        return set()
    try:
        return set(pd.read_csv(csv_path, usecols=['security'])['security'].astype(str))
    except Exception as e:
        print(f'  [resume] could not read {csv_path} ({repr(e)[:50]}) — treating as empty')
        return set()


def _append_csv(df, path):
    """Append DataFrame rows to a CSV, writing header on first write. Flush + fsync."""
    write_header = not os.path.exists(path)
    with open(path, 'a', newline='') as fh:
        df.to_csv(fh, header=write_header, index=False)
        fh.flush()
        os.fsync(fh.fileno())


# File-path helpers (analogous to V5_WF_CSV etc. in the HTBoost notebook)
def _wf_csv(h, out_dir):     return os.path.join(out_dir, f'cat_wf_H{h}__{MACHINE_ID}.csv')
def _pred_csv(h, out_dir):   return os.path.join(out_dir, f'cat_wf_preds_H{h}__{MACHINE_ID}.csv')
def _imp_csv(h, out_dir):    return os.path.join(out_dir, f'cat_wf_imps_H{h}__{MACHINE_ID}.csv')
def _imp_pkl(h, out_dir):    return os.path.join(out_dir, f'cat_wf_imps_H{h}__{MACHINE_ID}.pkl')
def _tuning_csv(h, out_dir): return os.path.join(out_dir, f'cat_tuning_log_H{h}__{MACHINE_ID}.csv')

print('Helper utilities and file-path helpers defined.')

### Load data and define the universe

Replicates the data-preparation steps of the HTBoost notebook and XGBoost script in order:
`load_data()` → `load_norway_raw()` (cache-first) → `add_norway_features()` → `add_macro_features()`.
Universe: NOR swap columns with ≥ `UNIVERSE_MIN_OBS` non-NaN observations.

In [ ]:
def load_and_augment_data():
    """Load Bloomberg data and augment with macro + Norway features.
    Steps are byte-identical to the XGBoost script's load_and_augment_data().
    """
    print('Loading Bloomberg data ...')
    df_raw = load_data()
    print(f'  df_raw shape: {df_raw.shape}   '
          f'({df_raw.index.min().date()} → {df_raw.index.max().date()})')

    print('Loading Norway data (cache-first) ...')
    start_str = df_raw.index.min().strftime('%Y-%m-%d')
    end_str   = df_raw.index.max().strftime('%Y-%m-%d')
    nor_raw, nor_report = load_norway_raw(start_str, end_str, NOR_CACHE, live=False)
    print_connectivity_report(nor_report)

    if not nor_raw.empty:
        df_raw = df_raw.join(nor_raw, how='left')
        df_raw = add_norway_features(df_raw, nor_raw)
        print(f'  Norway nor_* columns added: '
              f'{sum(1 for c in df_raw.columns if c.startswith("nor_"))}')
    else:
        print('  [WARN] Norway data unavailable — nor_* features will be NaN '
              '(CatBoost handles NaN natively; nor_* columns will have no predictive content)')

    print('Adding macro features ...')
    df_raw = add_macro_features(df_raw)
    print(f'  df_raw shape after augmentation: {df_raw.shape}')
    return df_raw


df_raw = load_and_augment_data()

_SWAP_PAT = re.compile(r'^[A-Z]+_\d+[WMY]$')
swap_cols = sorted(c for c in df_raw.columns if _SWAP_PAT.match(c))
obs       = df_raw[swap_cols].notna().sum()
universe  = sorted(obs[obs >= UNIVERSE_MIN_OBS].index.tolist())
universe  = [s for s in universe if s.rsplit('_', 1)[0] == 'NOR']
print(f'\nNorwegian swap universe ({len(universe)} securities): {universe}')
assert SMOKE_SEC in universe, f'{SMOKE_SEC} not in NOR universe — check data'

### Metrics contract — shared long-CSV schema

Every results row uses `compute_metrics_row` / `SHARED_COLS` from `src.evaluation.metrics` —
the single source of truth that all three models (HTBoost, XGBoost, CatBoost) share.
Rows produced here merge with `v5_metrics_long.csv` and `xgb_metrics_long*.csv`
with zero reconciliation.

In [ ]:
# Shared evaluation harness — single source of truth; NOT redefined here.
# compute_metrics_row, SHARED_COLS, apply_mtc_family, finalize_long_csv
# are already imported at the top of this notebook.
print(f'SHARED_COLS ({len(SHARED_COLS)}): {SHARED_COLS}')

### Cross-market PCA (fit on training rows only)

`fit_transform_xm_pca` from `src.features.pca`: standardise on training moments,
zero-fill NaN, fit PCA on training rows only, apply to both train and test.
k = smallest number of components explaining ≥ `XM_PCA_VAR` of training variance,
capped at `XM_PCA_KMAX`. Identical rule to HTBoost and XGBoost.

In [ ]:
# fit_transform_xm_pca is already imported at the top.
# Confirming the shared PCA rule matches config:
print(f'PCA rule: XM_PCA_ENABLE={XM_PCA_ENABLE}  var_target={XM_PCA_VAR}  kmax={XM_PCA_KMAX}')
print('fit_transform_xm_pca imported from src.features.pca (fit on TRAIN only).')

### CatBoost tuning and fitting

`_tune_and_fit_cat`: time-ordered inner-CV grid search + refit on full training window.

**Inner purge** (same fix as XGBoost script): the last H−1 rows of the inner-training
block are purged before early stopping, because y_{t,h} = r_{t+h} − r_t uses a future
rate whose value overlaps the inner-validation window. Degenerate-case guard: if the
purge would reduce the inner-training block below `MIN_TRAIN_OBS`, fall back to a
full-window fit with no early stopping (early-stopping label selection is skipped;
val_mse is still logged for the appendix).

In [ ]:
def _tune_and_fit_cat(y_tr, x_tr_df, y_te, x_te_df, H, seed=config.JULIA_SEED):
    """Time-ordered inner-CV grid search + refit on full training window.

    Inner validation: last INNER_VAL_FRAC of training rows (time-ordered).
    Grid search over CAT_PARAM_GRID with early stopping on the inner validation set.
    Selected hyperparameters are never exposed to the test window.

    CatBoost API notes vs XGBoost:
      iterations       ← n_estimators
      depth            ← max_depth (but oblivious trees: NOT equivalent, see CONFIG)
      l2_leaf_reg      ← reg_lambda
      rsm              ← colsample_bytree (random subspace method)
      min_data_in_leaf ← min_child_weight (integer count, not hessian sum)
      thread_count     ← n_jobs
      random_seed      ← random_state
      No bootstrap_type override → MVS default (see KNOWN DIVERGENCE 2 in title cell)
      subsample        = MVS data fraction; labelled cat_mvs_fraction in tuning log
      best_iteration_  is 0-indexed (same as XGBoost best_iteration) → n_est = +1
      No cat_features: per-security numeric-only configuration

    Returns
    -------
    yhat_tr      : ndarray   in-sample predictions on full training window
    yhat_te      : ndarray   OOS predictions on test window
    best_params  : dict      selected hyperparameters (CatBoost native names)
    best_n_est   : int       selected iterations
    tuning_rows  : list[dict]  per-combination log (cat_mvs_fraction, not subsample)
    imp_dict     : dict      {feature_name: PredictionValuesChange importance}
    """
    n           = len(y_tr)
    n_inner_val = max(MIN_TEST_OBS, int(n * INNER_VAL_FRAC))
    n_inner_tr  = n - n_inner_val

    # ── Inner purge (same fix as XGBoost script) ─────────────────────────────
    # Drop the last H-1 rows of the inner-training block so none of its labels
    # overlap the inner-validation window. y_{t,h} = r_{t+h} − r_t uses a future
    # rate: the last H-1 inner-train labels reach into the inner-validation period,
    # leaking future information into early-stopping's n_est selection.
    n_purge     = max(0, H - 1)
    n_it_purged = n_inner_tr - n_purge

    if n_it_purged >= MIN_TRAIN_OBS:
        # Normal path: purged inner-train is large enough for early stopping.
        X_it = x_tr_df.iloc[:n_it_purged].to_numpy(dtype=np.float64)
        y_it = y_tr[:n_it_purged]
        use_early_stopping = True
    else:
        # Degenerate-case guard: purge shrinks inner-train below MIN_TRAIN_OBS.
        # Falls back to full inner-train block with no early stopping.
        # val_mse is still logged for appendix; n_est defaults to N_TREES_MAX.
        X_it = x_tr_df.iloc[:n_inner_tr].to_numpy(dtype=np.float64)
        y_it = y_tr[:n_inner_tr]
        use_early_stopping = False

    X_iv       = x_tr_df.iloc[n_inner_tr:].to_numpy(dtype=np.float64)
    y_iv       = y_tr[n_inner_tr:]
    feat_names = list(x_tr_df.columns)   # saved before numpy conversion for importance mapping

    keys   = list(CAT_PARAM_GRID.keys())
    values = list(CAT_PARAM_GRID.values())

    best_mse    = np.inf
    best_params = None
    best_n_est  = N_TREES_MAX
    tuning_rows = []

    for combo in itertools.product(*values):
        params = dict(zip(keys, combo))

        # No bootstrap_type override → CatBoost uses MVS (CPU default).
        # subsample here = MVS data fraction (see KNOWN DIVERGENCE 2).
        # logging_level='Silent' suppresses all CatBoost output; do NOT also set
        # verbose= in the constructor or fit() — they are mutually exclusive in
        # CatBoost's param canonizer and raise CatBoostError if both are present.
        mdl_kw = dict(
            iterations=N_TREES_MAX,
            loss_function=LOSS_FUNCTION,
            random_seed=seed,
            thread_count=1,
            logging_level='Silent',
            allow_writing_files=False,
            **params,
        )

        if use_early_stopping:
            mdl = CatBoostRegressor(**mdl_kw)
            mdl.fit(X_it, y_it,
                    eval_set=(X_iv, y_iv),
                    early_stopping_rounds=EARLY_STOP_RND)
            # best_iteration_ is 0-indexed (same convention as XGBoost best_iteration).
            # Add 1 to convert to iteration count for the refit.
            n_est = int(getattr(mdl, 'best_iteration_', N_TREES_MAX - 1)) + 1
        else:
            mdl = CatBoostRegressor(**mdl_kw)
            mdl.fit(X_it, y_it)
            n_est = N_TREES_MAX

        yhat_iv = mdl.predict(X_iv)
        mse_iv  = float(np.mean((y_iv - yhat_iv) ** 2))

        # Rename subsample → cat_mvs_fraction in the log so reviewers can't
        # misread it as XGBoost-style Bernoulli subsampling.
        log_params = {
            ('cat_mvs_fraction' if k == 'subsample' else k): v
            for k, v in params.items()
        }
        tuning_rows.append({
            **log_params,
            'val_mse':           mse_iv,
            'best_n_estimators': n_est,
            'selected':          False,
        })

        if mse_iv < best_mse:
            best_mse    = mse_iv
            best_params = params.copy()
            best_n_est  = n_est

    # Flag the selected row in the tuning log.
    # tuning_rows uses cat_mvs_fraction for the subsample key; best_params uses subsample.
    for row in tuning_rows:
        if all(
            row.get('cat_mvs_fraction' if k == 'subsample' else k) == v
            for k, v in best_params.items()
        ):
            row['selected'] = True
            break

    # Refit on the full training window with the selected hyperparameters
    mdl_final = CatBoostRegressor(
        iterations=best_n_est,
        loss_function=LOSS_FUNCTION,
        random_seed=seed,
        thread_count=1,
        logging_level='Silent',
        allow_writing_files=False,
        **best_params,
    )
    X_tr_np = x_tr_df.to_numpy(dtype=np.float64)
    X_te_np = x_te_df.to_numpy(dtype=np.float64)
    mdl_final.fit(X_tr_np, y_tr)

    yhat_tr = mdl_final.predict(X_tr_np)
    yhat_te = mdl_final.predict(X_te_np)

    # Feature importance: PredictionValuesChange (analog of XGBoost gain importance).
    # Array is in training-feature order; zip with feat_names (saved before numpy conversion).
    imp_dict = {}
    try:
        imp_arr = mdl_final.get_feature_importance(type='PredictionValuesChange')
        imp_dict = {fn: float(v) for fn, v in zip(feat_names, imp_arr)}
    except Exception:
        pass

    return yhat_tr, yhat_te, best_params, best_n_est, tuning_rows, imp_dict


print('_tune_and_fit_cat defined.')

### Walk-forward runner

`_run_security_cat`: expanding-window walk-forward for one security at one horizon.
Replicates the fold logic of the XGBoost script's `_run_security_xgb` exactly:
same `WF_FOLDS`, same one-sided purge `purge_ts = test_start − BDay(H−1)`,
same PCA compression via `fit_transform_xm_pca` (fit on TRAIN only),
same `compute_metrics_row` / `SHARED_COLS` evaluation.

In [ ]:
def _run_security_cat(sec, df_raw, H, seed=config.JULIA_SEED, verbose=False):
    """Expanding-window walk-forward for one security at horizon H.
    Returns (rows, imp_records, tuning_rows_all, preds_list).
    """
    if sec not in df_raw.columns:
        return [], [], [], []
    panel = build_panel(df_raw, [sec], H)
    if len(panel) == 0:
        return [], [], [], []

    fc = [c for c in panel.columns if c not in META_COLS]
    country, tenor = sec.rsplit('_', 1)

    rows, imp_records, tuning_all, preds_list = [], [], [], []

    for fold_name, test_start, test_end, regime in WF_FOLDS:
        ts_ts = pd.Timestamp(test_start)
        te_ts = pd.Timestamp(test_end)
        # One-sided outer purge: identical to HTBoost and XGBoost (OVERLAP = H-1)
        purge_ts = ts_ts - pd.tseries.offsets.BDay(H - 1)

        tr = panel[panel['date'] < purge_ts].copy()
        te = panel[(panel['date'] >= ts_ts) & (panel['date'] <= te_ts)].copy()

        if len(tr) < MIN_TRAIN_OBS or len(te) < MIN_TEST_OBS:
            continue

        # PCA compression — fit on TRAIN only, apply identically to train + test.
        # Standardise on training moments, zero-fill NaN, fit PCA, apply to both.
        x_tr, x_te, pca_info = fit_transform_xm_pca(tr[fc], te[fc])
        feat_count = x_tr.shape[1]

        # Sanitize column names + replace ±inf; CatBoost handles NaN natively.
        x_tr_cat = _prepare_x_cat(x_tr)
        x_te_cat = _prepare_x_cat(x_te)

        y_tr_arr = tr['y'].to_numpy(float)
        y_te_arr = te['y'].to_numpy(float)

        yhat_tr, yhat_te, best_params, best_n_est, fold_tuning, imp_dict = \
            _tune_and_fit_cat(y_tr_arr, x_tr_cat, y_te_arr, x_te_cat, H, seed=seed)

        for trow in fold_tuning:
            trow.update({'security': sec, 'horizon': H,
                         'fold': fold_name, 'regime': regime})
        tuning_all.extend(fold_tuning)

        cfg_for_hash = {
            **best_params,
            'n_trees':        best_n_est,
            'loss_function':  LOSS_FUNCTION,
            'inner_val_frac': INNER_VAL_FRAC,
        }
        meta = dict(
            notebook=NOTEBOOK, run_ts=RUN_TS,
            model_kind=MODEL_KIND, is_pooled=IS_POOLED,
            validation_scheme='walk_forward', target_kind='level_change',
            security=sec, country=country, tenor=tenor,
            fold=fold_name, regime=regime,
            config_hash=_config_hash(cfg_for_hash, extra=FEAT_SPEC),
            feature_count=feat_count,
        )

        row_tr = compute_metrics_row(y_tr_arr, yhat_tr, H, {**meta, 'sample': 'train'})
        row_te = compute_metrics_row(y_te_arr, yhat_te, H, {**meta, 'sample': 'oos'})

        # Model-specific diagnostic columns — substituted for xgb_* / htb_* columns.
        # pca_k and xm_pca_evr are shared with both HTBoost and XGBoost.
        for row in (row_tr, row_te):
            row['pca_k']       = pca_info['k']
            row['xm_pca_evr']  = pca_info['evr_cum_k']
            row['cat_n_trees'] = best_n_est
            row['cat_depth']   = best_params.get('depth', -1)
            row['cat_lr']      = best_params.get('learning_rate', -1)

        rows.extend([row_tr, row_te])

        # Per-obs predictions (same schema as v5_wf_preds_H*.csv and xgb_wf_preds_H*.csv)
        pred_df = pd.DataFrame({
            'date':     te['date'].values,
            'security': sec,
            'tenor':    tenor,
            'horizon':  int(H),
            'regime':   regime,
            'scheme':   'walk_forward',
            'y_true':   y_te_arr,
            'y_pred':   np.asarray(yhat_te, float),
        })
        preds_list.append(pred_df)

        if imp_dict:
            imp_records.append({
                'security': sec, 'horizon': H,
                'fold': fold_name, 'regime': regime,
                'imp': imp_dict,
            })

        if verbose:
            s = _score(y_te_arr, yhat_te)
            if s:
                n_tr_total  = len(tr)
                n_inner_val = max(MIN_TEST_OBS, int(n_tr_total * INNER_VAL_FRAC))
                n_inner_tr  = n_tr_total - n_inner_val
                n_it_purged = max(0, n_inner_tr - (H - 1))
                use_es      = n_it_purged >= MIN_TRAIN_OBS
                ct_r2       = row_te.get('ct_r2_oos', float('nan'))
                print(f'    {fold_name:<12s}  '
                      f'outer_tr={n_tr_total:4d}  '
                      f'inner_tr_purged={n_it_purged:4d}  '
                      f'inner_val={n_inner_val:3d}  '
                      f'OOS={len(te):4d}  '
                      f'ES={str(use_es):<5s}  '
                      f'n_trees={best_n_est:4d}  '
                      f'DirAcc={s["dir_acc"]:.3f}  '
                      f'CT-R²={ct_r2:+.4f}  '
                      f'depth={best_params.get("depth")}  '
                      f'lr={best_params.get("learning_rate"):.3f}')

    return rows, imp_records, tuning_all, preds_list


print('_run_security_cat defined.')

### Horizon support gate and per-horizon sweep

`_horizon_supported`: identical logic to the XGBoost script — checks whether at least
one WF fold has enough data for the pilot security.

`_sweep_horizon`: walks forward across all tenors for one horizon, with resume support
(mirrors the XGBoost script's checkpoint structure: each completed security's fold-batch
is flushed and fsync'd immediately; resume skips completed securities).

In [ ]:
def _horizon_supported(df_raw, h):
    """Return True if at least one WF fold has ≥ MIN_TRAIN_OBS / MIN_TEST_OBS for SMOKE_SEC."""
    panel = build_panel(df_raw, [SMOKE_SEC], h)
    if len(panel) == 0:
        return False
    for _, ts, te, _ in WF_FOLDS:
        ts_ts = pd.Timestamp(ts)
        te_ts = pd.Timestamp(te)
        purge = ts_ts - pd.tseries.offsets.BDay(h - 1)
        ntr = (panel['date'] < purge).sum()
        nte = ((panel['date'] >= ts_ts) & (panel['date'] <= te_ts)).sum()
        if ntr >= MIN_TRAIN_OBS and nte >= MIN_TEST_OBS:
            return True
    return False


def _sweep_horizon(H_run, tenors, df_raw, out_dir, verbose=False):
    """Walk-forward sweep for one horizon across all tenors, with resume support.
    Mirrors the per-horizon checkpoint structure of the XGBoost script.
    """
    wf_csv_p   = _wf_csv(H_run, out_dir)
    pred_csv_p = _pred_csv(H_run, out_dir)
    imp_csv_p  = _imp_csv(H_run, out_dir)
    imp_pkl_p  = _imp_pkl(H_run, out_dir)
    tun_csv_p  = _tuning_csv(H_run, out_dir)

    done     = _done_secs(wf_csv_p)
    all_imps = []
    if os.path.exists(imp_pkl_p):
        try:
            with open(imp_pkl_p, 'rb') as f:
                all_imps = pickle.load(f)
        except Exception:
            pass
    if done:
        print(f'  [H={H_run}] resume: {len(done)} security(ies) already on disk — skipping them')

    failed = []
    for sec in tenors:
        if sec in done:
            continue
        t0 = time.time()
        try:
            rows, imp_recs, tuning_rows, preds = _run_security_cat(
                sec, df_raw, H_run, verbose=verbose)
        except Exception as e:
            print(f'  [H={H_run}] {sec}: FAILED  {repr(e)[:80]}')
            failed.append((H_run, sec))
            continue

        if rows:
            _append_csv(pd.DataFrame(rows), wf_csv_p)
        if preds:
            _append_csv(pd.concat(preds, ignore_index=True), pred_csv_p)
        if tuning_rows:
            _append_csv(pd.DataFrame(tuning_rows), tun_csv_p)
        if imp_recs:
            all_imps.extend(imp_recs)
            with open(imp_pkl_p, 'wb') as f:
                pickle.dump(all_imps, f)
                f.flush()
                os.fsync(f.fileno())
            imp_long = pd.DataFrame([
                {'security': r['security'], 'horizon': r['horizon'],
                 'fold': r['fold'], 'feature': feat, 'relevance': val}
                for r in all_imps
                for feat, val in (r['imp'] or {}).items()
            ])
            imp_long.to_csv(imp_csv_p, index=False)

        oos_rows = [r for r in rows if r.get('sample') == 'oos']
        da_str   = ', '.join(f'{r["dir_acc"]:.3f}' for r in oos_rows)
        print(f'  [H={H_run}] {sec:<10s}  folds={len(oos_rows)}  '
              f'OOS_DA=[{da_str}]  ({time.time()-t0:.1f}s)')

    if failed:
        print(f'  [H={H_run}] {len(failed)} failure(s): {failed}')
    return failed


print('_horizon_supported and _sweep_horizon defined.')

### Smoke test

Single end-to-end run: `NOR_10Y × H=21`, writes to `results/catboost_v5_nor/_smoke/`.
Validates plumbing (PCA, tuning, output schema) and prints per-fold diagnostics
including inner-train size after the H−1 purge, inner-val size, selected n_trees,
DirAcc, and CT-R²_OOS. Format is compared against both HTBoost and XGBoost output files.

In [ ]:
import glob as _glob

_out_dir = SMOKE_DIR if SMOKE_MODE else OUT_DIR
os.makedirs(_out_dir, exist_ok=True)
print(f'Output directory: {_out_dir}')
print(f'\n[SMOKE TEST]  {SMOKE_SEC} × H={SMOKE_H}')
print('─' * 75)

_t0 = time.time()
_rows, _imp_recs, _tuning_rows, _preds = _run_security_cat(
    SMOKE_SEC, df_raw, SMOKE_H, verbose=True)
_elapsed = time.time() - _t0

if not _rows:
    print('[SMOKE] No rows produced — check data and WF folds.')
else:
    _df = pd.DataFrame(_rows)
    _df.to_csv(_wf_csv(SMOKE_H, _out_dir), index=False)
    if _preds:
        pd.concat(_preds, ignore_index=True).to_csv(_pred_csv(SMOKE_H, _out_dir), index=False)
    if _tuning_rows:
        pd.DataFrame(_tuning_rows).to_csv(_tuning_csv(SMOKE_H, _out_dir), index=False)
    if _imp_recs:
        _imp_long = pd.DataFrame([
            {'security': r['security'], 'horizon': r['horizon'],
             'fold': r['fold'], 'feature': feat, 'relevance': val}
            for r in _imp_recs
            for feat, val in (r['imp'] or {}).items()
        ])
        _imp_long.to_csv(_imp_csv(SMOKE_H, _out_dir), index=False)

    _oos = _df[_df['sample'] == 'oos']
    _tr  = _df[_df['sample'] == 'train']
    print(f'\n{"─"*75}')
    print(f'[SMOKE RESULT]  {SMOKE_SEC}  H={SMOKE_H}  elapsed={_elapsed:.1f}s')
    print(f'  Folds : {len(_oos)} OOS folds  ({len(_tr)} train rows)')
    print(f'  Cols  : {list(_df.columns)}')
    print(f'\n  OOS per-fold summary (inner-train sizes printed above):')
    for _, _r in _oos.iterrows():
        print(f'    {_r["fold"]:<14s}  n={_r["n_obs"]:4.0f}  '
              f'DirAcc={_r["dir_acc"]:.3f}  CT-R²={_r["ct_r2_oos"]:+.4f}  '
              f'n_trees={_r["cat_n_trees"]:.0f}  depth={_r["cat_depth"]:.0f}  '
              f'lr={_r["cat_lr"]:.3f}')

    # ── Format check vs HTBoost ──────────────────────────────────────────────
    _v5_nor = os.path.join(_ROOT, 'results', 'v5_nor')
    _htb_ref = os.path.join(_v5_nor, f'v5_wf_H{SMOKE_H}__{MACHINE_ID}.csv')
    if not os.path.exists(_htb_ref):
        _cands = sorted(_glob.glob(os.path.join(_v5_nor, f'v5_wf_H{SMOKE_H}__*.csv')))
        _htb_ref = _cands[0] if _cands else None
    if _htb_ref and os.path.exists(_htb_ref):
        _htb_cols = set(pd.read_csv(_htb_ref, nrows=0).columns)
        _cat_cols = set(_df.columns)
        print(f'\n  ── Format check vs HTBoost ({os.path.basename(_htb_ref)}) ──')
        print(f'    Shared         : {len(_htb_cols & _cat_cols)} '
              f'(SHARED_COLS + pca_k + xm_pca_evr expected)')
        print(f'    HTBoost-only   : {sorted(_htb_cols - _cat_cols)}')
        print(f'    CatBoost-only  : {sorted(_cat_cols - _htb_cols)}')
        _sc_set = set(SHARED_COLS) | {"pca_k", "xm_pca_evr"}
        _missing = _sc_set - _cat_cols
        print(f'    SHARED_COLS+pca missing from CatBoost: {sorted(_missing)} '
              f'{"← OK" if not _missing else "← PROBLEM"}')
    else:
        print(f'  [format check vs HTBoost skipped — no v5_wf_H{SMOKE_H}__*.csv found]')

    # ── Format check vs XGBoost ──────────────────────────────────────────────
    _xgb_dir = os.path.join(_ROOT, 'results', 'xgb_v5_nor')
    _xgb_ref = None
    for _sd in [os.path.join(_xgb_dir, '_smoke'), _xgb_dir]:
        _c = sorted(_glob.glob(os.path.join(_sd, f'xgb_wf_H{SMOKE_H}__*.csv')))
        if _c:
            _xgb_ref = _c[0]; break
    if _xgb_ref and os.path.exists(_xgb_ref):
        _xgb_cols = set(pd.read_csv(_xgb_ref, nrows=0).columns)
        _cat_cols  = set(_df.columns)
        print(f'\n  ── Format check vs XGBoost ({os.path.basename(_xgb_ref)}) ──')
        print(f'    Shared         : {len(_xgb_cols & _cat_cols)} '
              f'(SHARED_COLS + pca_k + xm_pca_evr expected)')
        print(f'    XGBoost-only   : {sorted(_xgb_cols - _cat_cols)}  ← xgb_* expected')
        print(f'    CatBoost-only  : {sorted(_cat_cols - _xgb_cols)}  ← cat_* expected')
        _sc_set2 = set(SHARED_COLS) | {"pca_k", "xm_pca_evr"}
        _missing2 = _sc_set2 - _cat_cols
        print(f'    SHARED_COLS+pca missing from CatBoost: {sorted(_missing2)} '
              f'{"← OK" if not _missing2 else "← PROBLEM"}')
    else:
        print(f'  [format check vs XGBoost skipped — no xgb_wf_H{SMOKE_H}__*.csv found]')

    print(f'\nSmoke outputs written to: {_out_dir}')

### Walk-forward sweep — full run (SMOKE_MODE = False only)

Set `SMOKE_MODE = False` in the CONFIG cell to run the full sweep across all NOR tenors
and all horizons. Resume is automatic: completed (security, horizon) pairs are detected
from the per-horizon CSV and skipped.

In [ ]:
if not SMOKE_MODE:
    # Resolve horizon list (gate long horizons on data length, same as HTBoost / XGBoost)
    HORIZONS = list(H_GRID)
    for _h in H_GRID_LONG:
        if _horizon_supported(df_raw, _h):
            HORIZONS.append(_h)
            print(f'  long horizon H={_h}: data supports it → included')
        else:
            print(f'  long horizon H={_h}: insufficient data per fold → SKIPPED')
    HORIZONS = sorted(set(HORIZONS))
    SWEEP_TENORS = list(universe)

    print(f'\nSweep: {len(SWEEP_TENORS)} tenor(s) × {len(HORIZONS)} horizon(s)')
    print(f'  Tenors  : {SWEEP_TENORS}')
    print(f'  Horizons: {HORIZONS}')
    _n_combos = len(list(itertools.product(*CAT_PARAM_GRID.values())))
    print(f'  Grid    : {_n_combos} combinations/fold')
    print()

    _t0_sweep = time.time()
    _all_failed = []
    for _H_run in HORIZONS:
        print(f'H={_H_run} {"─"*50}')
        _failed = _sweep_horizon(_H_run, SWEEP_TENORS, df_raw, OUT_DIR, verbose=False)
        _all_failed.extend(_failed)

    # Assemble final metrics CSV + apply MTC (same family as HTBoost / XGBoost)
    _wf_parts = [pd.read_csv(_wf_csv(_h, OUT_DIR))
                 for _h in HORIZONS if os.path.exists(_wf_csv(_h, OUT_DIR))]
    if _wf_parts:
        _df_all = finalize_long_csv(pd.concat(_wf_parts, ignore_index=True))
        _N_wf, _b_wf, _h_wf = apply_mtc_family(
            _df_all, ['walk_forward'],
            'walk_forward:{horizon×tenor×regime}',
            model_kind=MODEL_KIND)
        _final_csv = os.path.join(OUT_DIR, f'cat_metrics_long__{MACHINE_ID}.csv')
        _df_all.to_csv(_final_csv, index=False)
        _elapsed_m = (time.time() - _t0_sweep) / 60
        print(f'\nSweep complete in {_elapsed_m:.1f} min')
        print(f'  rows={len(_df_all)}  cols={_df_all.shape[1]}')
        print(f'  Walk-forward MTC: N={_N_wf}  Bonferroni={_b_wf}  BH-FDR={_h_wf}')
        print(f'  Saved: {_final_csv}')
    if _all_failed:
        print(f'\n{len(_all_failed)} failure(s): {_all_failed}')
else:
    print('[Skipped] SMOKE_MODE=True — set SMOKE_MODE=False in CONFIG to run the full sweep.')